# Spatial panel models: a walkthrough

By the end of this lesson you will have built a balanced panel, fitted the
fixed-effects and random-effects families to it, compared them, and used the
panel specification tests to choose the spatial terms the data support.

It assumes you have been through
[the cross-sectional walkthrough](pedagogical_spatial_models.ipynb); the models
here are its panel counterparts. Work through the cells in order.

A panel stacks the same units across periods, which adds two things the
cross-sectional models did not have: unit effects $a_i$ and time effects
$\tau_t$. The `effects` argument controls which are included.

The equations for each class are in [Supported Models](../models.md).

In [ ]:
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from libpysal.graph import Graph

from neighbayes import dgp
from neighbayes.models import OLS, SAR, SDEM, SDM, SEM

az.style.use("arviz-variat")

## Build the panel

For pedagogy, we generate synthetic panel data from the `neighbayes.dgp` module on a regular polygon grid and then assemble a formula-ready DataFrame.


In [ ]:
# Generate base geometry via cross-sectional DGP, then simulate panel data via panel DGP.
rng = np.random.default_rng(2026)
xcols = ["poverty", "rev_rating", "num_spots", "crowded"]
ycol = "price_pp"

base_gdf = dgp.simulate_sar(
    n_side=20, seed=2026, create_gdf=True, geometry_type="polygon"
)
W = Graph.build_contiguity(base_gdf, rook=False).transform("r")
N = len(base_gdf)
T = 4
beta = np.array([1.5, -0.8, 0.6, 0.4, -0.5], dtype=float)

panel_sim = dgp.simulate_sar(
    N=N,
    T=T,
    rho=0.35,
    beta=beta,
    sigma=1.0,
    rng=rng,
    W=W,
    unit_effects="fixed",
    unit_effect_sd=0.5,
)

panel = pd.DataFrame(
    {
        "unit": panel_sim["unit"],
        "time": panel_sim["time"],
        ycol: panel_sim["y"],
    }
)
for j, name in enumerate(xcols, start=1):
    panel[name] = panel_sim["X"][:, j]

## A helper to keep the fits short

In [ ]:
def fit_panel_model(
    model_cls, formula, data, W, draws=2000, tune=1000, chains=2, seed=42, **design
):
    """Fit a panel model and return (model, summary, effects_df).

    Uses minimal MCMC settings for pedagogical demonstration.
    Increase draws/tune for real analyses.
    """
    m = model_cls(
        formula=formula,
        data=data,
        W=W,
        unit_col="unit",
        time_col="time",
        **design,
    )
    m.fit(
        draws=draws,
        tune=tune,
        chains=chains,
        random_seed=seed,
        progressbar=True,
    )
    summary = m.summary(round_to=3)
    effects_df = pd.DataFrame(m.impacts())
    return m, summary, effects_df


def diagnostics_table(idata, var_names):
    """Show key MCMC diagnostics for the given parameters."""
    cols = ["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]
    return az.summary(idata, var_names=var_names, round_to=3)[cols]


def show_trace(idata, var_names, title):
    """Plot trace plots for the given parameters."""
    az.plot_trace_dist(idata, var_names=var_names)
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

## Start without any spatial term

In [ ]:
formula = "price_pp ~ poverty + rev_rating + num_spots + crowded"
ols_panel, summary_ols, effects_ols = fit_panel_model(
    OLS, formula, panel, W, unit_effects="fixed", time_effects=True
)
display(summary_ols)
display(effects_ols)
display(diagnostics_table(ols_panel.inference_data, ["beta", "sigma"]))
show_trace(ols_panel.inference_data, ["sigma"], "OLS, two-way fixed effects: trace")

## Add a spatial lag on the outcome

In [ ]:
sar_panel, summary_sar, effects_sar = fit_panel_model(
    SAR, formula, panel, W, unit_effects="fixed", time_effects=True
)
display(summary_sar)
display(effects_sar)
display(diagnostics_table(sar_panel.inference_data, ["rho", "beta", "sigma"]))
show_trace(
    sar_panel.inference_data, ["rho", "sigma"], "SAR, two-way fixed effects: trace"
)

## Try spatial structure in the errors instead

In [ ]:
sem_panel, summary_sem, effects_sem = fit_panel_model(
    SEM, formula, panel, W, unit_effects="fixed", time_effects=True
)
display(summary_sem)
display(effects_sem)
display(diagnostics_table(sem_panel.inference_data, ["lam", "beta", "sigma"]))
show_trace(
    sem_panel.inference_data, ["lam", "sigma"], "SEM, two-way fixed effects: trace"
)

## Combine both channels

In [ ]:
sdm_panel, summary_sdm, effects_sdm = fit_panel_model(
    SDM, formula, panel, W, unit_effects="fixed", time_effects=True
)
display(summary_sdm)
display(effects_sdm)
display(diagnostics_table(sdm_panel.inference_data, ["rho", "beta", "sigma"]))
show_trace(
    sdm_panel.inference_data, ["rho", "sigma"], "SDM, two-way fixed effects: trace"
)

## The error-side combination

In [ ]:
sdem_panel, summary_sdem, effects_sdem = fit_panel_model(
    SDEM, formula, panel, W, unit_effects="fixed", time_effects=True
)
display(summary_sdem)
display(effects_sdem)
display(diagnostics_table(sdem_panel.inference_data, ["lam", "beta", "sigma"]))
show_trace(
    sdem_panel.inference_data, ["lam", "sigma"], "SDEM, two-way fixed effects: trace"
)

### Check the chains before comparing anything

Panel SAR/SDEM models inherit the slow-mixing behavior of $\rho$
(or $\lambda$) documented by Wolf, Anselin & Arribas-Bel (2018)
{cite:p}`wolf2018StochasticEfficiency`. The
`spatial_mcmc_diagnostic` helper checks ESS, sampler yield, $\hat{R}$,
and HPDI stability for the spatial scalar.


In [ ]:
from neighbayes.diagnostics import spatial_mcmc_diagnostic

spatial_mcmc_diagnostic(sdm_panel, emit_warnings=False).to_frame()

## Swap fixed effects for random effects

The random-effects models replace deterministic unit fixed effects with latent unit intercepts $\alpha_i \sim N(0, \sigma_\alpha^2)$. That lets the notebook estimate the variance of unit heterogeneity directly and compare FE and RE behavior on the same synthetic balanced panel.

### The non-spatial random-effects baseline

In [ ]:
ols_panel_re, summary_ols_re, effects_ols_re = fit_panel_model(
    OLS, formula, panel, W, unit_effects="random"
)
display(summary_ols_re)
display(effects_ols_re)
display(
    diagnostics_table(ols_panel_re.inference_data, ["beta", "sigma", "unit_effect_sd"])
)
show_trace(
    ols_panel_re.inference_data,
    ["sigma", "unit_effect_sd"],
    "OLS, random effects: trace",
)

### Random effects with a spatial lag

In [ ]:
sar_panel_re, summary_sar_re, effects_sar_re = fit_panel_model(
    SAR, formula, panel, W, unit_effects="random"
)
display(summary_sar_re)
display(effects_sar_re)
display(
    diagnostics_table(
        sar_panel_re.inference_data, ["rho", "beta", "sigma", "unit_effect_sd"]
    )
)
show_trace(
    sar_panel_re.inference_data,
    ["rho", "sigma", "unit_effect_sd"],
    "SAR, random effects: trace",
)

### Random effects with spatial errors

In [ ]:
sem_panel_re, summary_sem_re, effects_sem_re = fit_panel_model(
    SEM, formula, panel, W, unit_effects="random"
)
display(summary_sem_re)
display(effects_sem_re)
display(
    diagnostics_table(
        sem_panel_re.inference_data, ["lam", "beta", "sigma", "unit_effect_sd"]
    )
)
show_trace(
    sem_panel_re.inference_data,
    ["lam", "sigma", "unit_effect_sd"],
    "SEM, random effects: trace",
)

## Compare what you have fitted

Models with the same unit effects describe the same outcomes, so bridge sampling
can give each a marginal likelihood. Each entry below is the log Bayes factor for
the row's model over the column's, so positive values favor the row.

In [ ]:
from neighbayes.diagnostics import bayes_factor_compare_models

fixed = {
    "OLS": ols_panel,
    "SAR": sar_panel,
    "SEM": sem_panel,
    "SDM": sdm_panel,
    "SDEM": sdem_panel,
}
random = {"OLS": ols_panel_re, "SAR": sar_panel_re, "SEM": sem_panel_re}

for label, models in {"Fixed effects": fixed, "Random effects": random}.items():
    print(label)
    display(
        bayes_factor_compare_models(
            models, method="bridge", log=True, random_state=0
        ).round(1)
    )

Fixed and random effects cannot be compared this way. A fixed-effects likelihood
is the random-effects likelihood under a flat prior on the effects, which is
improper, so it has no marginal likelihood. `az.loo` can compare them, because
each model stores every observation's density given all the others, but it
measures prediction: random effects gain from pooling whether or not their
assumptions hold, so a better score does not show that their estimates are right.

Compare the estimates instead, which is the logic of a Hausman test. Unit effects
related to the covariates move both $\beta$ and $\rho$ under random effects; a
spatial pattern in the effects that the covariates do not explain moves $\rho$ and
leaves $\beta$ alone.

In [ ]:
fe_estimates = sar_panel.summary()
re_estimates = sar_panel_re.summary(var_names=["~unit_effect"])
shared = ["rho", *xcols]

contrast = pd.DataFrame(
    {
        "fixed": fe_estimates.loc[shared, "mean"],
        "random": re_estimates.loc[shared, "mean"],
    }
)
contrast["gap in fixed-effects SDs"] = (
    contrast["random"] - contrast["fixed"]
) / fe_estimates.loc[shared, "sd"]
display(contrast.round(3))
print(
    "largest gap: "
    f"{contrast['gap in fixed-effects SDs'].abs().max():.2f} fixed-effects SDs"
)

Gaps under about one posterior standard deviation give no reason to doubt the
random effects. Larger ones, in $\beta$ especially, point to unit effects that
move with the covariates, which only the fixed effects absorb.

## Let the tests pick the specification

The panel models run the cross-section's Lagrange-multiplier tests, written for
the panel and named with a `Panel-` prefix. Under fixed effects they score the
transformed likelihood the demeaned data give (Lee & Yu 2010); under random
effects the unit effects enter as coefficients carrying their prior.
`spatial_diagnostics()` runs the tests that apply to a fitted model, and
`spatial_diagnostics_decision()` reads them by the rule of the cross-sectional
walkthrough: a lone plain rejection adds its term, and the robust pair only
settles a double rejection. Fit the recommended model and ask again until the
tests stop rejecting.

The panel was simulated from a SAR with no $WX$ term, so the walk should end at
the SAR.

In [ ]:
def read_tests(model, alpha=0.05):
    """Show a fitted model's LM tests and say which reject."""
    diag = model.spatial_diagnostics()
    rejecting = diag.index[diag["p_value"] < alpha].tolist()
    name = type(model).__name__
    print(f"{name}: tests rejecting at {alpha}: {', '.join(rejecting) or 'none'}")
    print(
        f"{name}: recommended model: "
        f"{model.spatial_diagnostics_decision(alpha=alpha, format='model')}"
    )
    return diag[["statistic", "df", "p_value", "ci_lower", "ci_upper"]].round(3)

### Start from the fixed-effects OLS panel

The OLS panel's tests ask whether the outcome, the errors or the neighbors'
covariates carry spatial dependence once the unit and period effects are
removed.

In [ ]:
read_tests(ols_panel)

In [ ]:
ols_panel.spatial_diagnostics_decision()

### Refit, and ask again

The OLS panel's tree names the model to fit next, and that model tests the terms
it still leaves out; at the SAR panel, the neighbors' covariates (towards the
SDM) and the error process (towards SARAR, which neighbayes does not fit). The
cell below follows the recommendations through the fixed-effects models fitted
above until a model's tests stop rejecting.

In [ ]:
def walk(start, fitted, alpha=0.05, show=True):
    """Follow the recommendations from ``start`` through the fitted models."""
    model, path = start, [type(start).__name__]
    while True:
        nxt = model.spatial_diagnostics_decision(alpha=alpha, format="model")
        if nxt == path[-1]:
            break
        if nxt not in fitted:
            path.append(f"{nxt} (not fitted here)")
            break
        model = fitted[nxt]
        path.append(nxt)
        if show:
            print(f"--- at the {nxt} fit ---")
            display(read_tests(model, alpha))
            display(model.spatial_diagnostics_decision(alpha=alpha))
    if show:
        print(" → ".join(path))
    return path


fitted = {"SAR": sar_panel, "SEM": sem_panel, "SDM": sdm_panel, "SDEM": sdem_panel}
path = walk(ols_panel, fitted)

A random-effects fit runs the same tests through the same tree.

In [ ]:
read_tests(sar_panel_re)

## Where to go next

This lesson used small draw counts so it runs quickly. For real work raise
`draws` and `tune`, and if you see divergences or tree-depth warnings raise
`target_accept` to 0.95–0.99. It is also worth checking that your conclusions
survive a change of fixed-effects specification (`effects=0/1/2/3`) and of
priors.

- [How to run Bayesian LM specification tests](../how-to/bayesian_lmtests.ipynb)
- [How to fit spatial tobit panel models](../how-to/tobit_panel_models.ipynb) —
  panels with censored outcomes
- [How to set priors](../how-to/priors.ipynb) — the `Panel*Priors` classes
- [Supported Models](../models.md) — every panel class and its equations